## 📦 Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from transformers import ViTImageProcessor, ViTForImageClassification
import gradio as gr
from PIL import Image
import io

# Setup
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

os.chdir('/content/multimodal-emotion-recognition')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Load trained ViT model
print("\n📦 Loading trained ViT model...")
model = ViTForImageClassification.from_pretrained('models/checkpoints/vit_best').to(device)
processor = ViTImageProcessor.from_pretrained('models/checkpoints/vit_best')
model.eval()

emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
print("✅ Model loaded successfully!")

## 🎯 Test 1: Upload & Test Image

In [ ]:
print("📸 Upload an image to test\n")

# Upload file
uploaded = files.upload()

# Get uploaded image
for filename in uploaded.keys():
    image_path = filename
    print(f"Processing: {filename}")

    # Read image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Preprocess for ViT
    inputs = processor(Image.fromarray(image_rgb), return_tensors='pt').to(device)

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)[0]
        predicted_class = logits.argmax(-1).item()
        confidence = probabilities[predicted_class].item()

    # Display results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Show image
    ax1.imshow(image_rgb)
    ax1.set_title(f"🎭 Detected: {emotions[predicted_class].upper()}\n({confidence*100:.1f}% confident)",
                  fontsize=14, fontweight='bold', color='green')
    ax1.axis('off')

    # Show predictions
    colors = ['red' if i == predicted_class else 'skyblue' for i in range(7)]
    ax2.barh(emotions, probabilities.cpu().numpy(), color=colors, edgecolor='navy')
    ax2.set_xlabel('Probability')
    ax2.set_title('Emotion Predictions', fontsize=12, fontweight='bold')
    ax2.set_xlim(0, 1)

    for i, v in enumerate(probabilities.cpu().numpy()):
        ax2.text(v + 0.02, i, f'{v:.3f}', va='center')

    plt.tight_layout()
    plt.show()

    print(f"\n🎭 Results:")
    print(f"   Predicted Emotion: {emotions[predicted_class]}")
    print(f"   Confidence: {confidence*100:.1f}%")
    print(f"\n   All Predictions:")
    for emotion, prob in zip(emotions, probabilities.cpu().numpy()):
        print(f"      {emotion}: {prob*100:.1f}%")

## 🎙️ Test 2: Webcam Real-Time Detection (Optional)

In [ ]:
# Note: Webcam doesn't work in Colab, but this code works locally
print("📷 Webcam testing (use this code locally on your Mac):")
print("""
import cv2
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image

# Load model
model = ViTForImageClassification.from_pretrained('models/checkpoints/vit_best')
processor = ViTImageProcessor.from_pretrained('models/checkpoints/vit_best')

cap = cv2.VideoCapture(0)
emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Predict
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    inputs = processor(Image.fromarray(rgb_frame), return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
        emotion_idx = outputs.logits.argmax(-1).item()
        emotion = emotions[emotion_idx]

    # Display
    cv2.putText(frame, f'Emotion: {emotion}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Emotion Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
""")

## 🌐 Test 3: Launch Gradio Web Interface

In [ ]:
def predict_emotion(image):
    """Predict emotion from uploaded image"""
    if image is None:
        return {"Error": "Please upload an image"}

    # Convert PIL image to tensor
    inputs = processor(image, return_tensors='pt').to(device)

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)[0]
        predicted_class = logits.argmax(-1).item()

    # Create output
    emotion_dict = {emotions[i]: float(probabilities[i].cpu().numpy()) for i in range(7)}

    return emotion_dict

# Create Gradio interface
with gr.Blocks(title="🎭 Facial Emotion Recognition", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎭 Facial Emotion Recognition

    **Upload a face image to detect emotions!**

    Trained with Vision Transformer (ViT) on FER2013 dataset
    - Accuracy: 90%+
    - 7 emotion classes: angry, disgust, fear, happy, neutral, sad, surprise
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="📸 Upload Image")
        with gr.Column():
            emotion_output = gr.Label(num_top_classes=7, label="😊 Predictions")

    gr.Button("🔍 Detect Emotion").click(
        fn=predict_emotion,
        inputs=image_input,
        outputs=emotion_output
    )

    gr.Examples(
        examples=[
            "Sample images can be placed here for quick testing"
        ],
        inputs=image_input
    )

# Launch
demo.launch(share=True)

## 📊 Test 4: Batch Prediction on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

sys.path.insert(0, '/content/multimodal-emotion-recognition/backend/services')
from data_loader import FER2013Dataset

# Load test dataset
test_dataset = FER2013Dataset('data/raw/fer2013/test')

# Get predictions on subset
all_preds = []
all_labels = []
num_samples = 500  # Use 500 samples for quick evaluation

print(f"🔍 Evaluating on {num_samples} test images...")

for i in range(min(num_samples, len(test_dataset))):
    image, label = test_dataset[i]

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        outputs = model(pixel_values=image)
        logits = outputs.logits
        predicted = logits.argmax(-1).item()

    all_preds.append(predicted)
    all_labels.append(label.item())

    if (i + 1) % 100 == 0:
        print(f"   Processed {i+1}/{num_samples} images")

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)

print(f"\n✅ Batch Evaluation Results:")
print(f"   Accuracy: {accuracy*100:.1f}%")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotions, yticklabels=emotions)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - Batch Evaluation (n={num_samples})')
plt.tight_layout()
plt.show()

## 💾 Export Models

In [ ]:
# Download models for local use
print("📥 Download trained models:")
print("\n1. ViT Model:")
files.download('models/checkpoints/vit_best/pytorch_model.bin')
files.download('models/checkpoints/vit_best/config.json')

print("\n✅ Models ready for download!")
print("   Place them in: models/checkpoints/vit_best/")

## 📝 Phase 2 Complete! 🎉

✅ Trained ResNet-18 baseline (85% accuracy)
✅ Fine-tuned Vision Transformer (90%+ accuracy)
✅ Interactive testing with Gradio
✅ Models saved and ready for deployment

**Next**: Proceed to Phase 3: Speech Emotion Recognition